# Week 3: Semantic Entropy Probe (SEP) for Uncertainty Type Detection

## Inspiration

From [Semantic Entropy Probes (Kossen et al., 2024)](https://arxiv.org/abs/2406.15927):
> "SEPs are linear probes trained on the hidden states of LLMs to capture semantic entropy.
> Contrary to sampling-based detection, SEPs act directly on a single model hidden state."

## Key Insight

**Previous approaches failed because:**
- Token classification (hardcoded): Not robust to new languages/frameworks
- Embedding-based classification: Still needs seed tokens, biased toward Python/JS
- Our first probe: Detected "is this about code?" not "is uncertainty in code?"

**New approach:**
- Train probe to predict uncertainty TYPE directly
- No token classification needed at all
- Model's hidden state already encodes "what am I uncertain about"

## Architecture

```
Prompt: "import"          Prompt: "This function"
    │                           │
    ▼                           ▼
┌─────────────────────────────────────────┐
│           CodeLlama Forward Pass         │
│                                          │
│  Extract hidden state at layer L         │
│  (where L = model_layers / 3)            │
└─────────────────────────────────────────┘
    │                           │
    ▼                           ▼
  h ∈ R^4096                  h ∈ R^4096
    │                           │
    ▼                           ▼
┌─────────────────────────────────────────┐
│         Linear Probe (Logistic Reg)      │
│                                          │
│         P(code_uncertainty | h)          │
└─────────────────────────────────────────┘
    │                           │
    ▼                           ▼
  0.92 → CODE                 0.15 → LANG
```

---

## 1. Setup

In [ ]:
# Cell 1: Install
!pip install -q transformers torch accelerate scipy scikit-learn pandas matplotlib seaborn

In [ ]:
# Cell 2: Imports
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from scipy.stats import ttest_ind
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
print("Imports successful")

In [ ]:
# Cell 3: Load Model WITH hidden states output
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    output_hidden_states=True  # Important!
)
model.eval()

# Get model config
num_layers = model.config.num_hidden_layers
hidden_size = model.config.hidden_size

print(f"Model loaded on {model.device}")
print(f"Number of layers: {num_layers}")
print(f"Hidden size: {hidden_size}")
print(f"Vocabulary size: {len(tokenizer):,}")

## 2. Extract Hidden States

From the Probing-RAG paper:
> "Lower layers capture low-level info, higher layers capture abstract info.
> Position the prober after the 1/3 point of the model."

We'll test multiple layer positions:
- Early (1/4): Low-level features
- Middle (1/2): Balanced
- Late (3/4): High-level semantics
- Final: Output layer

In [ ]:
# Cell 4: Hidden state extraction function

def get_hidden_states(prompt: str, layer_indices: List[int] = None) -> Dict[int, np.ndarray]:
    """
    Extract hidden states at specified layers for the last token position.
    
    Args:
        prompt: Input text
        layer_indices: Which layers to extract (default: all)
    
    Returns:
        Dict mapping layer index to hidden state vector
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    
    # outputs.hidden_states is tuple of (num_layers + 1) tensors
    # Each tensor is (batch, seq_len, hidden_size)
    # Index 0 is embedding layer, 1 to num_layers are transformer layers
    
    hidden_states = {}
    
    if layer_indices is None:
        layer_indices = list(range(len(outputs.hidden_states)))
    
    for layer_idx in layer_indices:
        # Get last token's hidden state
        h = outputs.hidden_states[layer_idx][:, -1, :].squeeze().cpu().numpy()
        hidden_states[layer_idx] = h.astype(np.float32)
    
    return hidden_states

# Define layer positions to test
LAYER_POSITIONS = {
    'early': num_layers // 4,        # ~8 for 32-layer model
    'one_third': num_layers // 3,    # ~11 (recommended by Probing-RAG)
    'middle': num_layers // 2,       # ~16
    'late': (3 * num_layers) // 4,   # ~24
    'final': num_layers              # 32 (last layer)
}

print(f"Layer positions to test:")
for name, layer in LAYER_POSITIONS.items():
    print(f"  {name}: layer {layer}")

In [ ]:
# Cell 5: Test hidden state extraction

test_prompt = "import"
test_layers = list(LAYER_POSITIONS.values())

print(f"Testing hidden state extraction for: '{test_prompt}'")
hidden = get_hidden_states(test_prompt, test_layers)

for layer_idx, h in hidden.items():
    print(f"  Layer {layer_idx}: shape={h.shape}, norm={np.linalg.norm(h):.2f}, mean={h.mean():.4f}")

## 3. Training Data

We use our 20 labeled examples:
- 10 code uncertainty (label = 1)
- 10 language uncertainty (label = 0)

In [ ]:
# Cell 6: Define training examples

TRAIN_EXAMPLES = [
    # CODE UNCERTAINTY (label = 1)
    {'prompt': 'import', 'label': 1, 'desc': 'Uncertain which module'},
    {'prompt': 'from sklearn import', 'label': 1, 'desc': 'Uncertain sklearn module'},
    {'prompt': 'def process_data(df):\n    df.', 'label': 1, 'desc': 'Uncertain pandas method'},
    {'prompt': 'const [state, setState] = use', 'label': 1, 'desc': 'Uncertain React hook'},
    {'prompt': 'async function fetch_data() {\n    await', 'label': 1, 'desc': 'Uncertain async op'},
    {'prompt': 'model = tf.keras.', 'label': 1, 'desc': 'Uncertain Keras class'},
    {'prompt': 'app = FastAPI()\n@app.', 'label': 1, 'desc': 'Uncertain FastAPI decorator'},
    {'prompt': 'SELECT * FROM users WHERE', 'label': 1, 'desc': 'Uncertain SQL condition'},
    {'prompt': 'git ', 'label': 1, 'desc': 'Uncertain git command'},
    {'prompt': 'docker run -', 'label': 1, 'desc': 'Uncertain docker flag'},
    
    # LANGUAGE UNCERTAINTY (label = 0)
    {'prompt': 'This function', 'label': 0, 'desc': 'Uncertain which verb'},
    {'prompt': 'The algorithm is', 'label': 0, 'desc': 'Uncertain which adjective'},
    {'prompt': 'Code quality can be', 'label': 0, 'desc': 'Uncertain which verb'},
    {'prompt': 'Explain what this code', 'label': 0, 'desc': 'Uncertain which verb'},
    {'prompt': 'The main advantage of async programming is', 'label': 0, 'desc': 'Uncertain benefit'},
    {'prompt': 'TypeScript provides better', 'label': 0, 'desc': 'Uncertain improvement'},
    {'prompt': 'Recursion is useful when', 'label': 0, 'desc': 'Uncertain scenario'},
    {'prompt': 'REST APIs are designed to', 'label': 0, 'desc': 'Uncertain purpose'},
    {'prompt': 'The difference between let and const is', 'label': 0, 'desc': 'Uncertain explanation'},
    {'prompt': 'Unit tests help', 'label': 0, 'desc': 'Uncertain benefit'},
]

print(f"Training examples: {len(TRAIN_EXAMPLES)}")
print(f"  Code uncertainty: {sum(1 for e in TRAIN_EXAMPLES if e['label'] == 1)}")
print(f"  Language uncertainty: {sum(1 for e in TRAIN_EXAMPLES if e['label'] == 0)}")

In [ ]:
# Cell 7: Extract hidden states for all training examples

print("Extracting hidden states for all training examples...")
print("(This may take a few minutes)")

layer_indices = list(LAYER_POSITIONS.values())

# Store hidden states per layer
hidden_states_by_layer = {layer: [] for layer in layer_indices}
labels = []

for example in tqdm(TRAIN_EXAMPLES, desc="Extracting"):
    hidden = get_hidden_states(example['prompt'], layer_indices)
    
    for layer_idx in layer_indices:
        hidden_states_by_layer[layer_idx].append(hidden[layer_idx])
    
    labels.append(example['label'])

# Convert to numpy arrays
for layer_idx in layer_indices:
    hidden_states_by_layer[layer_idx] = np.array(hidden_states_by_layer[layer_idx])
    print(f"Layer {layer_idx}: {hidden_states_by_layer[layer_idx].shape}")

labels = np.array(labels)
print(f"\nLabels: {labels.shape}")

## 4. Train and Evaluate Probes

We train a separate logistic regression probe for each layer position.
Use Leave-One-Out cross-validation since we have limited data (20 examples).

In [ ]:
# Cell 8: Train probes at each layer position

print("Training probes at each layer position...")
print("Using Leave-One-Out cross-validation")
print("="*60)

results = []
best_accuracy = 0
best_layer = None

for layer_name, layer_idx in LAYER_POSITIONS.items():
    X = hidden_states_by_layer[layer_idx]
    y = labels
    
    # Normalize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Logistic Regression with LOO cross-validation
    probe = LogisticRegression(max_iter=1000, random_state=42)
    
    # Get predictions via LOO
    loo = LeaveOneOut()
    y_pred = cross_val_predict(probe, X_scaled, y, cv=loo)
    
    # Calculate accuracy
    accuracy = accuracy_score(y, y_pred)
    
    # Also get probabilities for analysis
    y_prob = cross_val_predict(probe, X_scaled, y, cv=loo, method='predict_proba')[:, 1]
    
    results.append({
        'layer_name': layer_name,
        'layer_idx': layer_idx,
        'accuracy': accuracy,
        'predictions': y_pred,
        'probabilities': y_prob,
    })
    
    print(f"\n{layer_name} (layer {layer_idx}):")
    print(f"  Accuracy: {accuracy:.0%} ({int(accuracy * len(y))}/{len(y)})")
    
    # Show errors
    errors = [(TRAIN_EXAMPLES[i], y_pred[i]) for i in range(len(y)) if y[i] != y_pred[i]]
    if errors:
        print(f"  Errors:")
        for ex, pred in errors:
            true_label = 'CODE' if ex['label'] == 1 else 'LANG'
            pred_label = 'CODE' if pred == 1 else 'LANG'
            print(f"    '{ex['prompt'][:30]}...' -> predicted {pred_label}, actual {true_label}")
    
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_layer = layer_name

print("\n" + "="*60)
print(f"Best layer: {best_layer} with {best_accuracy:.0%} accuracy")

In [ ]:
# Cell 9: Summary table

print("\nSUMMARY: Probe Accuracy by Layer")
print("="*60)
print(f"{'Layer':<15} {'Position':<10} {'Accuracy':<10}")
print("-"*35)

for r in results:
    print(f"{r['layer_name']:<15} {r['layer_idx']:<10} {r['accuracy']:.0%}")

# Compare to previous methods
print("\n" + "="*60)
print("COMPARISON TO PREVIOUS METHODS")
print("="*60)
print(f"{'Method':<30} {'Accuracy':<10} {'Robustness':<15}")
print("-"*55)
print(f"{'Hardcoded PMR (646 keywords)':<30} {'90%':<10} {'Low (fragile)':<15}")
print(f"{'Embedding PMR (20 seeds)':<30} {'90%':<10} {'Medium (OOD: 64%)':<15}")
print(f"{'SEP Probe (best layer)':<30} {f'{best_accuracy:.0%}':<10} {'High (learned)':<15}")

In [ ]:
# Cell 10: Visualize results

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Accuracy by layer
ax1 = axes[0, 0]
layer_names = [r['layer_name'] for r in results]
accuracies = [r['accuracy'] for r in results]
colors = ['green' if a >= 0.9 else 'orange' if a >= 0.7 else 'red' for a in accuracies]
bars = ax1.bar(layer_names, accuracies, color=colors, alpha=0.7)
ax1.axhline(y=0.9, color='blue', linestyle='--', label='Hardcoded PMR (90%)')
ax1.axhline(y=0.5, color='gray', linestyle='--', label='Random')
ax1.set_ylabel('Accuracy')
ax1.set_title('Probe Accuracy by Layer Position')
ax1.set_ylim(0, 1.1)
ax1.legend()
for bar, acc in zip(bars, accuracies):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{acc:.0%}', ha='center', fontsize=10)

# Plot 2: Best layer probe probabilities
ax2 = axes[0, 1]
best_result = [r for r in results if r['layer_name'] == best_layer][0]
probs = best_result['probabilities']
colors_prob = ['coral' if l == 1 else 'steelblue' for l in labels]
ax2.bar(range(len(probs)), probs, color=colors_prob, alpha=0.7)
ax2.axhline(y=0.5, color='black', linestyle='-', linewidth=2)
ax2.set_xlabel('Example Index')
ax2.set_ylabel('P(Code Uncertainty)')
ax2.set_title(f'Probe Predictions ({best_layer} layer)')
ax2.axvline(x=9.5, color='gray', linestyle='--', alpha=0.5)
ax2.text(4.5, 1.05, 'Code Unc (true)', ha='center', fontsize=10)
ax2.text(14.5, 1.05, 'Lang Unc (true)', ha='center', fontsize=10)

# Plot 3: Probe probability distribution
ax3 = axes[1, 0]
code_probs = probs[labels == 1]
lang_probs = probs[labels == 0]
ax3.hist(code_probs, bins=10, alpha=0.7, label='Code Uncertainty', color='coral')
ax3.hist(lang_probs, bins=10, alpha=0.7, label='Language Uncertainty', color='steelblue')
ax3.axvline(x=0.5, color='black', linestyle='-', linewidth=2)
ax3.set_xlabel('P(Code Uncertainty)')
ax3.set_ylabel('Count')
ax3.set_title('Probability Distribution by True Label')
ax3.legend()

# Plot 4: Layer comparison
ax4 = axes[1, 1]
layer_idxs = [r['layer_idx'] for r in results]
ax4.plot(layer_idxs, accuracies, 'o-', markersize=10, linewidth=2, color='purple')
ax4.fill_between(layer_idxs, accuracies, alpha=0.3, color='purple')
ax4.axhline(y=0.9, color='blue', linestyle='--', alpha=0.5)
ax4.set_xlabel('Layer Index')
ax4.set_ylabel('Accuracy')
ax4.set_title('Accuracy vs Layer Depth')
ax4.set_ylim(0, 1.1)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('week3_sep_probe_results.png', dpi=150)
plt.show()

## 5. Out-of-Distribution Testing

Test on examples the probe has never seen, including:
- Different programming languages (Rust, Go, Elixir)
- Niche libraries
- Different language patterns

In [ ]:
# Cell 11: Train final probe on ALL data using best layer

print(f"Training final probe on {best_layer} layer (all data)...")

best_layer_idx = LAYER_POSITIONS[best_layer]
X_train = hidden_states_by_layer[best_layer_idx]
y_train = labels

# Normalize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Train final probe
final_probe = LogisticRegression(max_iter=1000, random_state=42)
final_probe.fit(X_train_scaled, y_train)

print(f"Probe trained!")
print(f"Coefficients shape: {final_probe.coef_.shape}")

In [ ]:
# Cell 12: Define OOD test examples

OOD_EXAMPLES = [
    # Different programming languages (FAILED with embedding approach)
    {'prompt': 'fn main() {', 'label': 1, 'desc': 'Rust'},
    {'prompt': 'func main() {', 'label': 1, 'desc': 'Go'},
    {'prompt': 'defmodule MyApp do', 'label': 1, 'desc': 'Elixir'},
    {'prompt': 'fun main() =', 'label': 1, 'desc': 'SML/OCaml'},
    {'prompt': 'proc main {', 'label': 1, 'desc': 'Tcl'},
    
    # Niche libraries
    {'prompt': 'import Bio.', 'label': 1, 'desc': 'Biopython'},
    {'prompt': 'from scanpy import', 'label': 1, 'desc': 'Single-cell'},
    {'prompt': 'model = xgb.', 'label': 1, 'desc': 'XGBoost'},
    {'prompt': 'client = boto3.', 'label': 1, 'desc': 'AWS SDK'},
    
    # CLI tools
    {'prompt': 'kubectl get', 'label': 1, 'desc': 'Kubernetes'},
    {'prompt': 'terraform apply', 'label': 1, 'desc': 'Terraform'},
    {'prompt': 'ansible-playbook', 'label': 1, 'desc': 'Ansible'},
    
    # Language uncertainty (various styles)
    {'prompt': 'The architecture of this system', 'label': 0, 'desc': 'Technical doc'},
    {'prompt': 'In conclusion, the results show', 'label': 0, 'desc': 'Academic'},
    {'prompt': 'To summarize the key findings', 'label': 0, 'desc': 'Summary'},
    {'prompt': 'The primary benefit of using', 'label': 0, 'desc': 'Explanation'},
    {'prompt': 'This approach is preferred because', 'label': 0, 'desc': 'Justification'},
]

print(f"OOD test examples: {len(OOD_EXAMPLES)}")
print(f"  Code uncertainty: {sum(1 for e in OOD_EXAMPLES if e['label'] == 1)}")
print(f"  Language uncertainty: {sum(1 for e in OOD_EXAMPLES if e['label'] == 0)}")

In [ ]:
# Cell 13: Evaluate on OOD examples

print("ROBUSTNESS TEST: Out-of-Distribution Examples")
print("="*80)

ood_results = []

for example in tqdm(OOD_EXAMPLES, desc="Testing OOD"):
    # Get hidden state
    hidden = get_hidden_states(example['prompt'], [best_layer_idx])
    h = hidden[best_layer_idx].reshape(1, -1)
    
    # Scale
    h_scaled = scaler.transform(h)
    
    # Predict
    pred = final_probe.predict(h_scaled)[0]
    prob = final_probe.predict_proba(h_scaled)[0, 1]
    
    correct = pred == example['label']
    
    ood_results.append({
        'prompt': example['prompt'],
        'desc': example['desc'],
        'true_label': example['label'],
        'predicted': pred,
        'probability': prob,
        'correct': correct,
    })

# Print results
print("\nResults:")
print(f"{'Description':<20} {'Prompt':<25} {'P(code)':<10} {'Pred':<6} {'True':<6} {'Status'}")
print("-"*80)

for r in ood_results:
    pred_str = 'CODE' if r['predicted'] == 1 else 'LANG'
    true_str = 'CODE' if r['true_label'] == 1 else 'LANG'
    status = 'OK' if r['correct'] else 'WRONG'
    print(f"{r['desc']:<20} {r['prompt'][:25]:<25} {r['probability']:.2f}      {pred_str:<6} {true_str:<6} {status}")

ood_accuracy = sum(r['correct'] for r in ood_results) / len(ood_results)
print(f"\nOOD Accuracy: {ood_accuracy:.0%} ({sum(r['correct'] for r in ood_results)}/{len(ood_results)})")

In [ ]:
# Cell 14: Compare OOD performance to embedding approach

print("\nCOMPARISON: OOD Performance")
print("="*60)

# Embedding approach results (from previous notebook)
embedding_ood_accuracy = 0.64  # 7/11 correct

print(f"{'Method':<30} {'Main Test':<12} {'OOD Test':<12}")
print("-"*54)
print(f"{'Hardcoded PMR (646 keywords)':<30} {'90%':<12} {'N/A':<12}")
print(f"{'Embedding PMR (20 seeds)':<30} {'90%':<12} {'64%':<12}")
print(f"{'SEP Probe':<30} {f'{best_accuracy:.0%}':<12} {f'{ood_accuracy:.0%}':<12}")

improvement = ood_accuracy - embedding_ood_accuracy
print(f"\nOOD Improvement over Embedding: {improvement:+.0%}")

In [ ]:
# Cell 15: Analyze what the probe learned

print("\nANALYSIS: What did the probe learn?")
print("="*60)

# Get probe weights
weights = final_probe.coef_[0]

print(f"Probe weight statistics:")
print(f"  Mean: {weights.mean():.6f}")
print(f"  Std:  {weights.std():.6f}")
print(f"  Max:  {weights.max():.6f}")
print(f"  Min:  {weights.min():.6f}")

# Top positive and negative dimensions
top_positive = np.argsort(weights)[-10:][::-1]
top_negative = np.argsort(weights)[:10]

print(f"\nTop dimensions for CODE uncertainty:")
for idx in top_positive[:5]:
    print(f"  Dimension {idx}: weight = {weights[idx]:.4f}")

print(f"\nTop dimensions for LANGUAGE uncertainty:")
for idx in top_negative[:5]:
    print(f"  Dimension {idx}: weight = {weights[idx]:.4f}")

In [ ]:
# Cell 16: Save results

# Save probe
import pickle

probe_data = {
    'probe': final_probe,
    'scaler': scaler,
    'best_layer': best_layer,
    'best_layer_idx': best_layer_idx,
    'train_accuracy': best_accuracy,
    'ood_accuracy': ood_accuracy,
}

with open('sep_probe.pkl', 'wb') as f:
    pickle.dump(probe_data, f)

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv('week3_sep_probe_layer_results.csv', index=False)

ood_df = pd.DataFrame(ood_results)
ood_df.to_csv('week3_sep_probe_ood_results.csv', index=False)

print("Results saved!")
print(f"  - sep_probe.pkl (trained probe)")
print(f"  - week3_sep_probe_layer_results.csv")
print(f"  - week3_sep_probe_ood_results.csv")

## 6. Summary

### What We Did
1. Extracted hidden states from CodeLlama at multiple layer positions
2. Trained linear probes to predict uncertainty TYPE directly
3. Used Leave-One-Out cross-validation (20 examples)
4. Tested robustness on out-of-distribution examples

### Key Advantages of SEP Approach
- **No token classification**: Probe learns directly from hidden states
- **Model-learned features**: Uses what the model already knows
- **Language-agnostic**: Should generalize to any programming language
- **Simple**: Just a logistic regression on hidden states
- **Fast**: Single forward pass + linear probe

### Results
- Best layer: [see output]
- Main accuracy: [see output]
- OOD accuracy: [see output]

---